In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import pandas as pd
from sklearn.linear_model import TheilSenRegressor
from sklearn.linear_model import LinearRegression

from matplotlib import gridspec
import scipy.stats as stats

from scipy.stats import kstest, cramervonmises
import tensorflow as tf
import tensorflow_probability as tfp
import pykrige.kriging_tools as kt
from pykrige.ok import OrdinaryKriging
import time

from ipcc_colormap import *
from utils import *


import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Myriad Pro'
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 600

coastline = gpd.read_file('/home/mizu_home/xp53/nas/home/coastlines-split-SGregion/lines.shp')
mask = np.loadtxt('mask.txt')

ipcc_blue = (112.0/255, 160.0/255, 205.0/255, 1.0)
ipcc_orange = (196.0/255, 121.0/255, 0.0/255, 1.0)

tmp_cmap = ipcc_cmap()
tmp_cmap.read_rgb_data_from_excel()
;

2025-11-03 18:55:15.415066: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-03 18:55:15.489754: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


''

In [2]:
rain_obs = np.loadtxt('data/sta_monthly.csv')

In [ ]:
# rain_obs is 480 x 14: 480 months (40 years) x 14 stations
# Reshape to (40 years, 12 months, 14 stations) to group by calendar month
# rain_obs is 480 rows x 14 columns
# Each row is a month: row 0 is month 1, row 1 is month 2, ..., row 11 is month 12,
# row 12 is month 1 again, etc.
# We need to extract rows i, i+12, i+24, ... for each calendar month i
# tmp_rain_obs = np.mean(rain_obs, axis=1)
rain_reshaped = np.array([rain_obs[i::12, 0] for i in range(12)])  # Shape: (12 months, 40 years)
rain_reshaped = rain_reshaped.transpose(1, 0)  # Reshape to (40 years, 12 months)

# Create 4x3 subplots for 12 months
fig, axes = plt.subplots(4, 3, figsize=(15, 16))
axes = axes.flatten()

for month_idx in range(12):
    ax = axes[month_idx]
    
    # Get all 40 years of data for this calendar month from station 5
    # Shape: (40,) -> all 40 values for this month
    month_data = rain_reshaped[:, month_idx]
    
    # Calculate sample mean and standard deviation
    sample_mean = np.mean(month_data)
    sample_std = np.std(month_data, ddof=1)
    
    # Perform Shapiro-Wilk test for normality
    shapiro_stat, shapiro_p = stats.shapiro(month_data)
    
    # Plot histogram
    n, bins, patches = ax.hist(month_data, bins=8, density=True, alpha=0.7, 
                                color=ipcc_blue, edgecolor='black', linewidth=0.5)
    
    # Plot fitted Gaussian distribution
    x = np.linspace(month_data.min(), month_data.max(), 200)
    gaussian_pdf = stats.norm.pdf(x, sample_mean, sample_std)
    ax.plot(x, gaussian_pdf, color=ipcc_orange, linewidth=2, 
            label=f'N({sample_mean:.1f}, {sample_std:.1f}²)')
    
    # Labels and title
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    ax.set_title(f'{month_names[month_idx]}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Rainfall (mm)', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    
    # Add Shapiro-Wilk test results to legend
    ax.legend([f'N({sample_mean:.1f}, {sample_std:.1f}²)',
               f'Shapiro-Wilk: p={shapiro_p:.3f}'], fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
plt.savefig('rain_hist.png')